# Naturalezza: rating di tutti i riscrittori con ordini bilanciati

Questo notebook riprende le **riscritture già prodotte** dallo studio su 126 originali e prepara una valutazione nuova della naturalezza. Non esegue nuovamente riscrittori, detector o modelli semantici.

**Impostazioni Kaggle:** Accelerator **None / CPU**, Internet **On**. La GPU non serve: il giudice viene eseguito da Groq. Per le chiamate, collegare il Secret `GROQ_API_KEY` al notebook senza incollare la chiave nelle celle.

Si mostrano insieme i quattro output dello stesso originale, con nomi anonimi, e si chiede un voto **da 1 a 5 per ciascuno**, consentendo voti uguali. Si usano quattro ordini Williams bilanciati, ognuno ripetuto tre volte: **12 chiamate per originale**. Ogni metodo occupa ogni posizione tre volte; sono bilanciate anche le adiacenze dirette.

Il disegno misura la sensibilità all'ordine e la stabilità delle ripetizioni: **non garantisce che il bias sparisca**. Copre quattro dei 24 ordini possibili. A temperatura zero le ripetizioni possono essere identiche. I voti sono giudizi di un LLM, da confrontare successivamente con annotazioni umane; non misurano la fedeltà semantica.

Partire da `MODE = "smoke"` e `RUN_API = False`. Il piano e il numero di chiamate si controllano prima di attivare le richieste. Dopo il test iniziale si può usare `MODE = "study"` in una cartella distinta, mantenendo fissi modello, prompt e regole di analisi.


## 1. Configurazione

Con i dati attuali, lo studio comprende 126 originali: **123 hanno tutti e quattro gli output validi**. I tre originali incompleti rimangono nel registro delle esclusioni; non vengono sostituiti. Il piano completo prevede quindi **1.476 chiamate**, oltre a eventuali tentativi di trasporto, sempre entro il limite impostato.

`MAX_POST_ATTEMPTS` conta tutti i tentativi POST già effettuati, anche dopo una ripresa. `MIN_REQUEST_INTERVAL` rallenta le richieste per ridurre le raffiche; non supera né garantisce le quote del provider. La disponibilità del modello viene verificata prima delle nuove chiamate, senza sostituzioni automatiche.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, stat, subprocess, sys, tempfile, time, zipfile

MODE = "smoke"                   # "smoke" oppure "study"
RUN_API = False                  # True soltanto quando si vogliono effettuare le chiamate
RUN_SUFFIX = "v1"
REPO_REF = "main"
JUDGE_MODEL = "qwen/qwen3.8-27b"
HUMANIZERS = ["qwen_humanize", "qwen14b_humanize", "dipper", "dipper_context3"]
REPETITIONS = 3
SEED = 42
BOOTSTRAP = 2000
MAX_POST_ATTEMPTS = 50 if MODE == "smoke" else 1650
MIN_REQUEST_INTERVAL = 4.0
INPUT_ARCHIVE = ""               # Vuoto: usa lo ZIP dello studio già incluso nella repository
RESUME_ZIP = ""                  # Facoltativo: percorso dello ZIP precedente in /kaggle/input/...

assert MODE in {"smoke", "study"}
assert len(HUMANIZERS) == len(set(HUMANIZERS)) == 4
assert REPETITIONS >= 1 and MAX_POST_ATTEMPTS >= 1 and MIN_REQUEST_INTERVAL >= 0
assert Path("/kaggle/working").is_dir(), "Questo notebook è predisposto per Kaggle."
MAX_SOURCES = 3 if MODE == "smoke" else 126
RUN = Path("/kaggle/working") / f"humanization_naturalness_panel_{MODE}_{RUN_SUFFIX}"
REPO = Path("/kaggle/working/.humanization_naturalness_code/Humanization")
RUN.mkdir(parents=True, exist_ok=True)
probe = RUN / ".write_probe"
probe.write_text("ok")
probe.unlink()
MPL_CACHE = Path("/kaggle/working/.humanization_naturalness_cache/matplotlib")
MPL_CACHE.mkdir(parents=True, exist_ok=True)
cache_probe = MPL_CACHE / ".write_probe"
cache_probe.write_text("ok")
cache_probe.unlink()
os.environ["MPLCONFIGDIR"] = str(MPL_CACHE)
print("Modalità:", MODE, "| cartella:", RUN)
print("API attive:", RUN_API, "| limite totale POST:", MAX_POST_ATTEMPTS)
print("Giudice:", JUDGE_MODEL, "| intervallo minimo richieste:", MIN_REQUEST_INTERVAL, "secondi")


## 2. Ripresa facoltativa di una sessione precedente

Per riprendere dopo un'interruzione o dopo l'esaurimento di una quota, caricare lo ZIP dei risultati come input Kaggle e impostare `RESUME_ZIP`. Usare la **stessa modalità e gli stessi parametri scientifici**. I file esistenti non vengono sovrascritti e le risposte già salvate non vengono richieste nuovamente.

Il codice della sessione precedente viene recuperato tramite il commit registrato. Per cambiare protocollo o modello, usare un nuovo `RUN_SUFFIX`; non mescolare valutazioni con prompt diversi.


In [ ]:
frozen_settings = {
    "mode": MODE, "model": JUDGE_MODEL, "systems": HUMANIZERS,
    "repetitions": REPETITIONS, "seed": SEED, "max_sources": MAX_SOURCES,
}
if RESUME_ZIP:
    resume_path = Path(RESUME_ZIP)
    assert resume_path.is_file(), f"ZIP da riprendere assente: {resume_path}"
    with zipfile.ZipFile(resume_path) as archive:
        assert sum(item.file_size for item in archive.infolist()) <= 512 * 2**20, "ZIP di risultati troppo grande."
        for item in archive.infolist():
            assert not any(part in {".", ".."} for part in item.filename.split("/")), "Componenti ZIP non ammesse."
            relative = Path(item.filename)
            assert not relative.is_absolute() and ".." not in relative.parts and "\\" not in item.filename, "Percorso ZIP non valido."
            assert relative.parts and relative.parts[0] != ".", "Percorso ZIP vuoto."
            assert not stat.S_ISLNK(item.external_attr >> 16), "I collegamenti simbolici non sono ammessi."
            target = (RUN / relative).resolve()
            assert target == RUN.resolve() or RUN.resolve() in target.parents, "Percorso esterno alla cartella del run."
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            assert target.suffix.lower() in {".json", ".jsonl", ".csv", ".png"}, f"File inatteso nello ZIP: {relative}"
            content = archive.read(item)
            if target.exists():
                assert target.read_bytes() == content, f"File diverso già presente: {relative}. Usare una nuova cartella."
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(content)
    print("Risultati precedenti ripristinati senza sovrascritture.")

runtime_path = RUN / "runtime_info.json"
saved_runtime = json.loads(runtime_path.read_text()) if runtime_path.exists() else None
if saved_runtime:
    assert saved_runtime["settings"] == frozen_settings, "Parametri diversi dal run precedente: scegliere un nuovo RUN_SUFFIX."
    assert saved_runtime["requested_ref"] == REPO_REF, "REPO_REF diverso: ripristinare il riferimento precedente o cambiare cartella."
    print("Commit congelato per la ripresa:", saved_runtime["code_commit"])
else:
    print("Nuovo run: nessuna risposta precedente da recuperare.")


## 3. Codice e sole dipendenze necessarie

Si usa la repository **Humanization**, in una cartella scrivibile sotto `/kaggle/working`. Non si creano ambienti virtuali e non si installano i requisiti dei modelli GPU. Le quattro librerie necessarie sono normalmente già presenti nell'immagine Kaggle; vengono installate soltanto quelle mancanti.


In [ ]:
repository_url = "https://github.com/FrancescoM00/Humanization.git"
REPO.parent.mkdir(parents=True, exist_ok=True)
fresh_clone = not REPO.exists()
if fresh_clone:
    subprocess.run(["git", "clone", "--depth", "1", repository_url, str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), f"Cartella esistente non riconosciuta come repository: {REPO}"
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)
    assert not dirty.strip(), "La copia del codice ha modifiche locali: conservarle e scegliere un'altra cartella REPO."
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip()
    assert remote.removesuffix(".git") == repository_url.removesuffix(".git"), "Repository diversa da quella prevista."

selected_ref = saved_runtime["code_commit"] if saved_runtime else REPO_REF
if not (fresh_clone and selected_ref == "main"):
    subprocess.run(["git", "fetch", "--depth", "1", "origin", selected_ref], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert (REPO / "scripts/judge_naturalness_panel.py").is_file(), "Il riferimento scelto non contiene il nuovo script."
print("Repository:", REPO)
print("Commit utilizzato:", CODE_COMMIT)

minimal_packages = {
    "requests": "requests>=2.32,<3", "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.2,<4", "matplotlib": "matplotlib>=3.8,<4",
}
missing = [requirement for module, requirement in minimal_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
import pandas as pd
from IPython.display import FileLink, Image, display
from importlib.metadata import version
LIBRARY_VERSIONS = {name: version(name) for name in minimal_packages}
print("Librerie:", LIBRARY_VERSIONS)


## 4. Recuperare le riscritture già fatte

Dallo ZIP dello studio si legge **solo** `rewriting/rewrites.jsonl`: nessun download di checkpoint, nessuna nuova generazione. Il file viene conservato negli input del nuovo run con la sua impronta SHA-256.


In [ ]:
input_archive = (Path(INPUT_ARCHIVE) if INPUT_ARCHIVE else
                 REPO / "exports/kaggle_v2_study/humanization_v2_study_protocol_v2.zip")
assert input_archive.is_file(), f"Archivio degli output precedente non trovato: {input_archive}"
with zipfile.ZipFile(input_archive) as archive:
    member = "rewriting/rewrites.jsonl"
    info = archive.getinfo(member)
    assert info.file_size <= 32 * 2**20, "Dimensione inattesa per il file delle riscritture."
    input_bytes = archive.read(member)
REWRITES = RUN / "inputs/rewrites.jsonl"
REWRITES.parent.mkdir(parents=True, exist_ok=True)
if REWRITES.exists():
    assert REWRITES.read_bytes() == input_bytes, "Le riscritture differiscono da quelle già registrate: non vengono sovrascritte."
else:
    REWRITES.write_bytes(input_bytes)
input_sha256 = hashlib.sha256(input_bytes).hexdigest()
rows = [json.loads(line) for line in input_bytes.decode().splitlines() if line.strip()]
source_ids = {str(row["source_id"]) for row in rows}
print("Testi registrati:", len(rows), "| originali distinti:", len(source_ids))
print("SHA-256 input:", input_sha256)

runtime = {
    "settings": frozen_settings, "requested_ref": REPO_REF, "code_commit": CODE_COMMIT,
    "input_sha256": input_sha256, "input_archive_name": input_archive.name,
    "python": sys.version.split()[0], "library_versions": LIBRARY_VERSIONS,
}
if saved_runtime:
    assert saved_runtime["code_commit"] == CODE_COMMIT
    assert saved_runtime["input_sha256"] == input_sha256
else:
    runtime_path.write_text(json.dumps(runtime, ensure_ascii=False, indent=2) + "\n")

SCRIPT = REPO / "scripts/judge_naturalness_panel.py"
def panel_command(execute=False):
    command = [sys.executable, str(SCRIPT), "--rewrites", str(REWRITES),
               "--output", str(RUN), "--model", JUDGE_MODEL,
               "--max-sources", str(MAX_SOURCES), "--repetitions", str(REPETITIONS),
               "--seed", str(SEED), "--max-requests", str(MAX_POST_ATTEMPTS),
               "--bootstrap", str(BOOTSTRAP),
               "--min-request-interval", str(MIN_REQUEST_INTERVAL)]
    for humanizer in HUMANIZERS:
        command.extend(["--humanizer", humanizer])
    if execute:
        command.append("--execute")
    return command

def load_json(name):
    path = RUN / name
    return json.loads(path.read_text()) if path.is_file() else {}

def read_rows(name):
    path = RUN / name
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.is_file() else []


## 5. Preparare e controllare il piano — senza chiave o chiamate

Questa cella è sempre offline rispetto al giudice. La selezione delle fonti precede il controllo dei fallimenti; un originale incompleto viene contato e non sostituito. Si registrano tutti gli ordini e le ripetizioni prima di richiedere i voti.


In [ ]:
offline_env = dict(os.environ)
offline_env.pop("GROQ_API_KEY", None)
subprocess.run(panel_command(execute=False), cwd=REPO, env=offline_env, check=True)
plan = load_json("judge_plan.json")
eligibility = load_json("judge_eligibility.json")
print("Fonti selezionate:", eligibility.get("n_selected_sources", "consultare judge_eligibility.json"))
print("Pannelli validi previsti:", plan.get("n_sources"))
print("Chiamate pianificate:", plan.get("expected_completion_requests"))
print("Limite POST totale impostato:", MAX_POST_ATTEMPTS)
print("Le ripetizioni e gli ordini sono nel piano; nessun voto è stato inventato.")
if eligibility.get("failures"):
    print("Fonti escluse per output mancanti o non validi:", len(eligibility["failures"]))
    display(pd.DataFrame(eligibility["failures"]).head(10))
assert "GROQ_API_KEY" not in offline_env


## 6. Chiamate al giudice e ripresa

Per avviare le chiamate, impostare `RUN_API = True` nella configurazione e rieseguire la configurazione e questa cella. Per lo studio completo cambiare prima `MODE` in `"study"` e rieseguire le celle 1–5: avrà una cartella e un piano distinti.

Le risposte e i tentativi vengono salvati progressivamente. La stessa cella riprende il piano saltando le richieste già risposte; non ricomincia da zero. Una quota esaurita o un errore possono lasciare lo stato **partial**: scaricare comunque lo ZIP e riprendere quando il servizio lo consente. Se il limite totale POST è stato raggiunto, aumentarlo esplicitamente soltanto se si desiderano altre richieste. Una risposta non valida resta registrata e non viene ripetuta fino a ottenere un voto favorevole.

Il tempo dipende dal servizio, dalla pausa fra richieste e dalle quote. Il test iniziale permette una stima pratica della velocità; nessuna GPU Kaggle può superare i limiti dell'API.


In [ ]:
planned_ids = {item["request_id"] for item in plan.get("requests", [])}
answered_ids = {row["request_id"] for row in read_rows("judgments.jsonl")}
pending_before = planned_ids - answered_ids
attempts_before = len(read_rows("api_attempts.jsonl"))

if RUN_API and pending_before:
    from kaggle_secrets import UserSecretsClient
    api_secret = UserSecretsClient().get_secret("GROQ_API_KEY")
    assert isinstance(api_secret, str) and api_secret.strip(), "Secret GROQ_API_KEY non disponibile."
    api_env = dict(os.environ)
    api_env["GROQ_API_KEY"] = api_secret.strip()
    started = time.monotonic()
    try:
        outcome = subprocess.run(panel_command(execute=True), cwd=REPO, env=api_env, check=False)
    finally:
        api_env.pop("GROQ_API_KEY", None)
        del api_secret
    elapsed = time.monotonic() - started
    if outcome.returncode != 0:
        print("Il comando si è fermato con codice", outcome.returncode,
              ": controllare l'errore sopra e conservare lo ZIP dei file disponibili.")
    new_attempts = len(read_rows("api_attempts.jsonl")) - attempts_before
    print("Tempo di questa esecuzione, minuti:", round(elapsed / 60, 2))
    if new_attempts:
        seconds_per_attempt = elapsed / new_attempts
        remaining = len(planned_ids - {row["request_id"] for row in read_rows("judgments.jsonl")})
        print("Secondi medi per nuovo tentativo, inclusa la pausa:", round(seconds_per_attempt, 2))
        if remaining:
            print("Stima lineare dei minuti per le richieste mancanti:", round(remaining * seconds_per_attempt / 60, 1),
                  "— non include futuri blocchi o attese per quota.")
elif not RUN_API:
    print("RUN_API=False: preparazione e analisi offline soltanto; nessuna chiave letta.")
else:
    print("Nessuna richiesta nuova: tutte le risposte del piano sono già registrate, oppure non ci sono pannelli validi.")

manifest = load_json("judge_run.json")
for field in ["status", "blocked_reason", "generation_post_attempts", "expected_completion_requests",
              "completed_valid_requests", "n_complete_source_panels", "n_primary_sources", "token_usage"]:
    if field in manifest:
        print(field + ":", manifest[field])


## 7. Leggere i risultati

- `system_summary.csv`: rating medio, distribuzione e intervalli per riscrittore.
- `source_system_scores.csv`: aggregati per originale e metodo.
- `paired_differences.csv`: confronti appaiati fra riscrittori sugli stessi originali.
- `position_effects.csv`: effetto della posizione dopo aver centrato i voti entro originale e metodo.
- `repeatability.csv` e `order_sensitivity.csv`: variazioni tra ripetizioni identiche e tra ordini diversi.
- `source_coverage.csv` e `coverage.csv`: pannelli completi, incompleti ed esclusi, anche per dominio e lunghezza.
- `ratings.csv`: voti individuali; `judgments.jsonl` e `api_attempts.jsonl`: risposte e tentativi API.

Il confronto primario usa soltanto pannelli con **tutte le chiamate valide**, così ogni metodo ha la stessa esposizione alle posizioni. I risultati parziali restano disponibili, con la copertura dichiarata. Le unità indipendenti sono gli originali, non le chiamate ripetute. Nessuna scheda vuota viene interpretata come un voto.


In [ ]:
for name in ["system_summary.csv", "paired_differences.csv", "position_effects.csv", "coverage.csv", "repeatability.csv"]:
    path = RUN / name
    if not path.is_file():
        continue
    try:
        frame = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        continue
    print(name, "— righe:", len(frame))
    if len(frame):
        display(frame.head(16))

figures = sorted(RUN.glob("*.png"))
for figure in figures:
    print(figure.name)
    display(Image(filename=str(figure)))
if not read_rows("judgments.jsonl"):
    print("Non sono ancora presenti risposte del giudice: questa esecuzione contiene soltanto il piano.")


## 8. Scaricare i file

Scaricare questo archivio anche se il run è parziale. Include piano, input, voti disponibili, tabelle e grafici. Non contiene pesi, ambiente virtuale, cartella del codice o chiavi. Per continuare in una nuova sessione, caricarlo come input Kaggle e indicarne il percorso in `RESUME_ZIP`.


In [ ]:
archive_path = Path(shutil.make_archive(str(RUN), "zip", root_dir=RUN))
with zipfile.ZipFile(archive_path) as archive:
    assert not any(Path(name).name == ".env" for name in archive.namelist())
    assert not any(name.startswith((".git/", ".venv/")) for name in archive.namelist())
print("Archivio da scaricare:", archive_path)
print("Dimensione MiB:", round(archive_path.stat().st_size / 2**20, 2))
display(FileLink(str(archive_path)))
